# Анализ временного ряда ETTh1

Ноутбук подготовлен для Google Colab и GitHub. Он показывает все результаты внутри ячеек: таблицы, метрики и графики. Файлы в папки не сохраняются.


## Запуск в Google Colab

Сначала выполните ячейку ниже, затем запускайте ноутбук сверху вниз.


In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install pandas numpy matplotlib scikit-learn statsmodels

print("Среда готова.")


## Импорты и загрузка данных


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)

LOCAL_DATA = Path("data/etth1_ot_prepared.csv")
ETTH1_URL = "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv"
RANDOM_STATE = 42


def load_etth1():
    if LOCAL_DATA.exists():
        df = pd.read_csv(LOCAL_DATA)
        source = str(LOCAL_DATA)
    else:
        raw = pd.read_csv(ETTH1_URL)
        df = raw[["date", "OT"]].rename(columns={"date": "ds", "OT": "y"})
        source = ETTH1_URL

    df["ds"] = pd.to_datetime(df["ds"])
    df["y"] = pd.to_numeric(df["y"], errors="coerce")
    df = df.dropna(subset=["ds", "y"]).sort_values("ds").drop_duplicates("ds")
    df = df.tail(3500).reset_index(drop=True)
    return df, source


df, source = load_etth1()
print(f"Источник данных: {source}")
print(f"Размер ряда: {df.shape[0]} строк")
display(df.head())


## Проверка качества и EDA


In [ ]:
quality = pd.DataFrame({
    "metric": ["start", "end", "rows", "missing_y", "duplicate_dates", "frequency_mode"],
    "value": [
        df["ds"].min(),
        df["ds"].max(),
        len(df),
        int(df["y"].isna().sum()),
        int(df["ds"].duplicated().sum()),
        df["ds"].diff().mode().iloc[0],
    ],
})
display(quality)

display(df["y"].describe().to_frame("y"))

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df["ds"], df["y"], linewidth=1)
ax.set_title("ETTh1: целевой ряд OT")
ax.set_xlabel("Дата")
ax.set_ylabel("OT")
ax.grid(alpha=0.25)
plt.show()


In [ ]:
eda = df.copy()
eda["hour"] = eda["ds"].dt.hour
eda["dayofweek"] = eda["ds"].dt.dayofweek

hour_profile = eda.groupby("hour")["y"].agg(["count", "mean", "std", "min", "max"]).round(3)
dow_profile = eda.groupby("dayofweek")["y"].agg(["count", "mean", "std", "min", "max"]).round(3)

print("Профиль по часам")
display(hour_profile)
print("Профиль по дням недели")
display(dow_profile)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
hour_profile["mean"].plot(ax=axes[0], marker="o", title="Среднее значение по часу")
dow_profile["mean"].plot(ax=axes[1], marker="o", title="Среднее значение по дню недели")
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

adf_stat, adf_pvalue, *_ = adfuller(df["y"])
diff = df["y"].diff(24).dropna()
adf_diff_stat, adf_diff_pvalue, *_ = adfuller(diff)
display(pd.DataFrame({
    "series": ["original", "seasonal_diff_24"],
    "adf_stat": [adf_stat, adf_diff_stat],
    "p_value": [adf_pvalue, adf_diff_pvalue],
}).round(5))


## Прогнозирование


In [ ]:
HORIZON = 24
train = df.iloc[:-HORIZON].copy()
test = df.iloc[-HORIZON:].copy()


def make_features(data):
    out = data.copy()
    for lag in [1, 2, 3, 24, 48, 168]:
        out[f"lag_{lag}"] = out["y"].shift(lag)
    for window in [24, 48, 168]:
        out[f"roll_mean_{window}"] = out["y"].shift(1).rolling(window).mean()
        out[f"roll_std_{window}"] = out["y"].shift(1).rolling(window).std()
    out["hour"] = out["ds"].dt.hour
    out["dayofweek"] = out["ds"].dt.dayofweek
    out["month"] = out["ds"].dt.month
    return out.dropna().reset_index(drop=True)


FEATURES = [
    "lag_1", "lag_2", "lag_3", "lag_24", "lag_48", "lag_168",
    "roll_mean_24", "roll_std_24", "roll_mean_48", "roll_std_48",
    "roll_mean_168", "roll_std_168", "hour", "dayofweek", "month",
]


def metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAPE_%": np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100,
    }


def recursive_forecast(model, history, horizon, scaler=None):
    hist = history.copy().reset_index(drop=True)
    preds = []
    step = hist["ds"].diff().mode().iloc[0]
    for _ in range(horizon):
        next_ds = hist["ds"].iloc[-1] + step
        row = pd.DataFrame({"ds": [next_ds], "y": [np.nan]})
        candidate = pd.concat([hist, row], ignore_index=True)
        feats = make_features(candidate).tail(1)
        x = feats[FEATURES].to_numpy()
        if scaler is not None:
            x = scaler.transform(x)
        pred = float(model.predict(x)[0])
        preds.append(pred)
        hist.loc[len(hist)] = {"ds": next_ds, "y": pred}
    return np.array(preds)


feature_train = make_features(train)
X_train = feature_train[FEATURES].to_numpy()
y_train = feature_train["y"].to_numpy()
y_test = test["y"].to_numpy()

results = []
forecast_table = test[["ds", "y"]].rename(columns={"y": "actual"}).reset_index(drop=True)

seasonal_naive = train["y"].iloc[-24:].to_numpy()
results.append({"group": "Baseline", "model": "SeasonalNaive_24", **metrics(y_test, seasonal_naive)})
forecast_table["SeasonalNaive_24"] = seasonal_naive

for name, model, use_scaler in [
    ("LinearRegression", LinearRegression(), False),
    ("Ridge_alpha_10", Ridge(alpha=10.0), False),
    ("RandomForest", RandomForestRegressor(n_estimators=120, random_state=RANDOM_STATE), False),
    ("MLP_small", MLPRegressor(hidden_layer_sizes=(32,), max_iter=400, random_state=RANDOM_STATE), True),
]:
    scaler = StandardScaler() if use_scaler else None
    X_fit = scaler.fit_transform(X_train) if scaler else X_train
    model.fit(X_fit, y_train)
    pred = recursive_forecast(model, train, HORIZON, scaler=scaler)
    results.append({"group": "Model", "model": name, **metrics(y_test, pred)})
    forecast_table[name] = pred

results_df = pd.DataFrame(results).sort_values("MAE").reset_index(drop=True)
display(results_df.round(4))
display(forecast_table.round(4))

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(train["ds"].tail(7 * 24), train["y"].tail(7 * 24), label="train tail")
ax.plot(test["ds"], y_test, marker="o", label="actual")
for col in results_df["model"].head(3):
    ax.plot(forecast_table["ds"], forecast_table[col], marker="o", label=col)
ax.set_title("Сравнение прогнозов на тестовом окне")
ax.grid(alpha=0.25)
ax.legend()
plt.show()


## Анализ аномалий


In [ ]:
anom = df.copy()
stl = STL(anom["y"], period=24, robust=True).fit()
anom["residual"] = stl.resid
q1, q3 = anom["residual"].quantile([0.25, 0.75])
iqr = q3 - q1
anom["Seasonal_IQR"] = (anom["residual"] < q1 - 1.5 * iqr) | (anom["residual"] > q3 + 1.5 * iqr)

rolling_mean = anom["y"].rolling(48, min_periods=24).mean()
rolling_std = anom["y"].rolling(48, min_periods=24).std()
z = (anom["y"] - rolling_mean) / rolling_std
anom["Rolling_Z"] = z.abs() > 3

iso_features = make_features(anom)[FEATURES]
iso = IsolationForest(contamination=0.03, random_state=RANDOM_STATE)
iso_pred = iso.fit_predict(iso_features)
anom["IsolationForest"] = False
anom.loc[iso_features.index, "IsolationForest"] = iso_pred == -1

summary = pd.DataFrame({
    "method": ["Seasonal_IQR", "Rolling_Z", "IsolationForest"],
    "anomalies": [int(anom["Seasonal_IQR"].sum()), int(anom["Rolling_Z"].sum()), int(anom["IsolationForest"].sum())],
})
display(summary)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(anom["ds"], anom["y"], linewidth=1, label="y")
points = anom[anom["Seasonal_IQR"]]
ax.scatter(points["ds"], points["y"], color="crimson", s=18, label="Seasonal_IQR")
ax.set_title("Найденные аномалии")
ax.grid(alpha=0.25)
ax.legend()
plt.show()


## Итог

Ноутбук завершает полный цикл: загрузка данных, EDA, проверка стационарности, сравнение моделей прогноза и поиск аномалий. Все результаты отображаются в ячейках.
